# Entrenamiento DenseNet121 - PlantVillage
**Arquitectura:** DenseNet121 con pesos preentrenados en PlantVillage (AP_pesos.h5)

**Dataset:** PlantVillage (Kaggle) - carpetas por clase

**Clases:** 38 enfermedades de plantas

> **Nota:** Se usan pesos ya fine-tuneados en PlantVillage, por lo que NO se necesita conexión a internet para descargar pesos de ImageNet.

## 1. Imports

In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import time
from itertools import cycle

import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report
)

print('TensorFlow version:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

2026-05-26 04:57:21.523365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779771441.691193      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779771441.738677      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779771442.126172      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779771442.126204      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779771442.126207      23 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2. Rutas de archivos

Ajusta las rutas según los nombres de tus datasets en Kaggle.

In [2]:
# ── AJUSTA ESTAS RUTAS ──────────────────────────────────────────────────────
# Ruta al dataset de imágenes (carpeta color con subcarpetas por clase)
DATA_DIR = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color'

# Ruta al archivo de pesos preentrenados
WEIGHTS_PATH = '/kaggle/input/datasets/yennybelhancco/pesos1/AP_pesos.h5'
# ─────────────────────────────────────────────────────────────────────────────

# Verificar dataset
if os.path.exists(DATA_DIR):
    categories = sorted(os.listdir(DATA_DIR))
    print(f'Dataset encontrado: {DATA_DIR}')
    print(f'Total de clases: {len(categories)}')
    for i, cat in enumerate(categories):
        count = len(glob.glob(os.path.join(DATA_DIR, cat, '*.[jJ][pP][gG]')))
        print(f'  {i+1:2d}. {cat:55s} → {count} imágenes')
else:
    print(f'ERROR: No se encontró {DATA_DIR}')
    print('Datasets disponibles en /kaggle/input/:')
    for d in os.listdir('/kaggle/input'):
        print(f'  /kaggle/input/{d}')

print()
if os.path.exists(WEIGHTS_PATH):
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Pesos encontrados: {WEIGHTS_PATH} ({size_mb:.1f} MB)')
else:
    print(f'ERROR: No se encontraron los pesos en {WEIGHTS_PATH}')

Dataset encontrado: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
Total de clases: 38
   1. Apple___Apple_scab                                      → 630 imágenes
   2. Apple___Black_rot                                       → 621 imágenes
   3. Apple___Cedar_apple_rust                                → 275 imágenes
   4. Apple___healthy                                         → 1645 imágenes
   5. Blueberry___healthy                                     → 1502 imágenes
   6. Cherry_(including_sour)___Powdery_mildew                → 1052 imágenes
   7. Cherry_(including_sour)___healthy                       → 854 imágenes
   8. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot      → 513 imágenes
   9. Corn_(maize)___Common_rust_                             → 1192 imágenes
  10. Corn_(maize)___Northern_Leaf_Blight                     → 985 imágenes
  11. Corn_(maize)___healthy                                  → 1162 imágenes
  12. Grape___Black_rot                    

## 3. Parámetros

In [3]:
# DenseNet121 usa 224x224
BATCH_SIZE    = 32
IMG_SIZE      = 224     # ← DenseNet121 usa 224x224
EPOCHS        = 20
LEARNING_RATE = 0.0001  # LR bajo porque los pesos ya están bien ajustados

num_classes = len(categories)
print(f'Número de clases: {num_classes}')
print(f'Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Épocas máximas: {EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')

Número de clases: 38
Tamaño de imagen: 224x224
Batch size: 32
Épocas máximas: 20
Learning rate: 0.0001


In [4]:
import shutil, os

DATA_DIR_ORIGINAL = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color'
WORK_DIR = '/kaggle/working/dataset_completo'
BACKGROUND_SRC = '/kaggle/input/datasets/yennybelhancco/background-withou-leaves/Background_without_leaves'

# Copiar sobre el directorio existente
shutil.copytree(DATA_DIR_ORIGINAL, WORK_DIR, dirs_exist_ok=True)
print(f'Clases copiadas: {len(os.listdir(WORK_DIR))}')

# Copiar Background
BACKGROUND_DST = os.path.join(WORK_DIR, 'Background_without_leaves')
shutil.copytree(BACKGROUND_SRC, BACKGROUND_DST, dirs_exist_ok=True)
print(f'Background copiado: {len(os.listdir(BACKGROUND_DST))} imágenes')

# Actualizar variables
DATA_DIR = WORK_DIR
categories = sorted(os.listdir(DATA_DIR))
num_classes = len(categories)
print(f'Total clases: {num_classes}')

Clases copiadas: 38


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/yennybelhancco/background-withou-leaves/Background_without_leaves'

## 4. Generadores de datos

In [ ]:
from tensorflow.keras.applications.densenet import preprocess_input

# Generador de entrenamiento con aumentación
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← agregar
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.3,
    shear_range=0.2,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← agregar
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='training',
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    class_mode='categorical',
    shuffle=False,
    seed=42
)

class_names = list(train_gen.class_indices.keys())
print(f'Imágenes de entrenamiento: {train_gen.samples}')
print(f'Imágenes de validación:    {val_gen.samples}')
print(f'Clases detectadas:         {len(class_names)}')

## 5. Cargar modelo con pesos preentrenados

En lugar de descargar DenseNet121 desde internet, cargamos directamente  que ya contiene DenseNet121 fine-tuneado en PlantVillage.

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2  # ← agregar

print('Descargando DenseNet121 con pesos ImageNet...')

base_model = DenseNet121(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # Congelar base

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dropout(0.5),                                              # 0.3 → 0.5
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),  # ← agregar l2
    Dropout(0.3),                                              # ← agregar segundo dropout
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Modelo listo.')
model.summary()

## 6. Preparar para fine-tuning

Descongelamos las últimas capas de DenseNet121 para continuar el entrenamiento con el dataset.

In [ ]:
# La base DenseNet121 es la primera capa del modelo
base_model = model.layers[0]

# Congelar todas las capas de la base primero
base_model.trainable = True

# Descongelar solo las últimas 50 capas para fine-tuning (DenseNet121 ~427 capas)
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompilar con learning rate bajo (importante después de descongelar capas)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Contar parámetros
trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
print(f'Parámetros entrenables:    {trainable:,}')
print(f'Parámetros no entrenables: {non_trainable:,}')

## 7. Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        min_delta=0.010
        verbose=1,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        'best_densenet121.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print('Callbacks configurados:')
print('  - EarlyStopping (patience=3)')
print('  - ModelCheckpoint → best_densenet121.h5')
print('  - ReduceLROnPlateau (factor=0.5, patience=2)')

## 8. Entrenamiento (fine-tuning)

In [ ]:
print('='*60)
print('INICIANDO FINE-TUNING - DenseNet121')
print('='*60)

start_time = time.time()

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

elapsed = time.time() - start_time
print(f'Tiempo total de entrenamiento: {elapsed/60:.1f} minutos')

## 9. Evaluación

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('/kaggle/working/best_densenet121.h5')
val_loss, val_acc = model.evaluate(val_gen, verbose=1)
print(f'Val accuracy: {val_acc:.4f}')
print(f'Val loss: {val_loss:.4f}')

In [ ]:
val_loss, val_acc = model.evaluate(val_gen, verbose=1)
print(f'Validation Loss:     {val_loss:.4f}')
print(f'Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)')

## 10. Curvas de entrenamiento

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'],     label='Train Loss',  color='blue')
plt.plot(history.history['val_loss'], label='Val Loss',    color='red', linestyle='--')
plt.title('DenseNet121 - Pérdida', fontsize=13, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Loss')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'],     label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Val Accuracy',   color='red', linestyle='--')
plt.title('DenseNet121 - Exactitud', fontsize=13, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('densenet121_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Predicciones y métricas

In [ ]:
print('Generando predicciones...')
val_gen.reset()
y_pred_proba = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = val_gen.classes

print(f'Muestras evaluadas: {len(y_pred)}')
print(f'Accuracy manual:    {np.mean(y_true == y_pred)*100:.2f}%')

## 12. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(22, 18))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=class_names, yticklabels=class_names,
    vmin=0, vmax=1
)
plt.title('DenseNet121 - Matriz de Confusión Normalizada', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('Clase Real', fontsize=12)
plt.xlabel('Clase Predicha', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('densenet121_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Reporte de clasificación

In [ ]:
report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
print('='*80)
print('REPORTE DE CLASIFICACIÓN - DenseNet121')
print('='*80)
print(report)

with open('densenet121_classification_report.txt', 'w') as f:
    f.write('REPORTE DE CLASIFICACIÓN - DenseNet121\n')
    f.write('='*80 + '\n')
    f.write(report)
print('Guardado: densenet121_classification_report.txt')

## 14. Curvas ROC

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import numpy as np

# Binarizar etiquetas
y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))

# Calcular ROC por clase
fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average
fpr['micro'], tpr['micro'], _ = roc_curve(y_true_bin.ravel(), y_pred_proba.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

# Macro-average
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(num_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= num_classes
fpr['macro'] = all_fpr
tpr['macro'] = mean_tpr
roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

print(f'AUC Macro: {roc_auc["macro"]:.4f}')
print(f'AUC Micro: {roc_auc["micro"]:.4f}')

In [ ]:
plt.plot(fpr['micro'], tpr['micro'], color='deeppink', linestyle=':', lw=4,
         label=f'Micro-avg (AUC={roc_auc["micro"]:.3f})')
plt.plot(fpr['macro'], tpr['macro'], color='navy', linestyle=':', lw=4,
         label=f'Macro-avg (AUC={roc_auc["macro"]:.3f})')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=13)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=13)
plt.title('DenseNet121 - Curvas ROC', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('densenet121_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC Macro: {roc_auc["macro"]:.4f}')
print(f'AUC Micro: {roc_auc["micro"]:.4f}')

## 15. Guardar modelo y métricas

In [ ]:
# Guardar modelo final
model.save('densenet121_plantvillage_finetuned.h5')
print('Modelo guardado: densenet121_plantvillage_finetuned.h5')

report_dict = classification_report(
    y_true, y_pred, target_names=class_names,
    output_dict=True, zero_division=0
)

metricas = {
    'arquitectura': 'DenseNet121',
    'pesos_base': 'ImageNet',
    'val_accuracy': float(val_acc),
    'val_loss': float(val_loss),
    'epocas_entrenadas': len(history.history['accuracy']),
    'macro_avg_precision': report_dict['macro avg']['precision'],
    'macro_avg_recall':    report_dict['macro avg']['recall'],
    'macro_avg_f1':        report_dict['macro avg']['f1-score'],
    'weighted_avg_f1':     report_dict['weighted avg']['f1-score'],
    'auc_macro':           float(roc_auc['macro']),
    'auc_micro':           float(roc_auc['micro']),
    'num_clases':          num_classes,
    'img_size':            IMG_SIZE,
    'parametros_totales':  int(model.count_params())
}

with open('densenet121_metricas.json', 'w') as f:
    json.dump(metricas, f, indent=2)

print('\nMétricas guardadas: densenet121_metricas.json')
print('\n=== RESUMEN FINAL ===')
for k, v in metricas.items():
    print(f'  {k}: {v}')